### Import pysammos

In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "8"
print(f">>> Numba is using {os.environ['NUMBA_NUM_THREADS']} cores")

>>> Numba is using 8 cores


In [2]:
import numpy as np

In [3]:
from pysammos.utils.config_loader import load_config
from pysammos.coarse_graining import CoarseGraining

Hello from pysammos
Loading macroscopic_fields package...
Loading data_read package...
Loading particle_phase package...
Loading spatial_weights package...
Loading neighbour_search package...
Loading grid_generation package...
Loading data_handle package...
Loading data_write package...


### Run Coarse - Graining Workflow

Initialise the Coarse Graining class

In [8]:
# Load the configuration from the ini file
cfg = load_config("config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=cfg["dt"],
    DEM_keymap=cfg["key_mapping"],
    grid_info=cfg["grid_info"],
    weight_function=cfg["smoothing_function"],
    fields_to_export=cfg["fields_to_export"],
    ignore_phases=cfg["partialignore"],
    h5_output=cfg["h5_output"],
    vkthdf_output=cfg["vkthdf_output"],
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

-------------------- config.ini file read --------------------
Output path already exists: ./PysammosCG/
  
-------------------- CoarseGraining class initialised --------------------


In [9]:
CG.DEM_keymap

{'Global_ID': 'Particle_ID',
 'Particle_Velocity': 'Velocity',
 'Particle_Diameter': 'Diameter',
 'Particle_Density': 'Density',
 'Particle_Volume': 'Volume',
 'Particle_Mass': 'Mass',
 'Particle_Radius': None,
 'Coordination_Number': 'Coordination',
 'Particle_i_ID': 'Particle_ID_1',
 'Particle_j_ID': 'Particle_ID_2',
 'Force_ij': 'FORCE_CHAIN_FN',
 'Contact_ij': 'FORCE_CHAIN_CONTACT_POINT'}

Option (A) Run all at once:

In [6]:
#CG.run()

Option (B) Run step by step:

In [10]:
# 1. Load the size-relevant particle data for the first time step
Bounds_t0, Diameter_t0, Density_t0, Mass_t0, GlobalID_t0 = CG.data_sampling()

XML-based PolyData detected.


In [11]:
# 2. Calculate the particle size range
CG.get_particle_size_statistics(Diameter_t0, Mass_t0)
print(">> Particle size statistics: ") 
print("       d43: ", CG.d43)
print("       dmax: ", CG.dmax)
print("       d50: ", CG.d50)
print("       d32: ", CG.d32)
print("       drms: ", CG.drms)

>> Particle size statistics: 
       d43:  0.000831585079243432
       dmax:  0.0048
       d50:  0.0008441205
       d32:  0.0008210011942357392
       drms:  0.0008051802416799283


In [12]:
# 3. Get the phases
CG.get_particle_phases(Diameter_t0, Density_t0, GlobalID_t0, 8)
print(">> Phases: ")
print("       Diameters: ", CG.phases[:,0])
print("       Densities: ", CG.phases[:,1])

Ignoring phases. Using mean density and d50 as phase.
>> Phases: 
       Diameters:  [0.00084412]
       Densities:  [2500.]


In [13]:
# 4. Calculate the CG grid spacing
CG.set_resolution(CG.d43) # here you can input different number, to make w and c bigger or smaller 
print(">> Coarse Graining resolution: ")
print("       c:", CG.c)
print("       w:", CG.w)

>> Coarse Graining resolution: 
       c: 0.001247377618865148
       w: 0.000623688809432574


In [14]:
# 5. Generate the CG grid
CG.generate_grid()
print(">> Grid: ")
print("       Grid Points: ", CG.GridPoints.shape, "First Point: ", CG.GridPoints[0])
print("       Nodes: ", CG.Nodes)
print("       Spacing: ", CG.Spacing)

Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.01, 0.07], y = [0.01, 0.07], z = [0.035, 0.045], x_transect = None, y_transect = None, z_transect = None
>> Grid: 
       Grid Points:  (172872, 3) First Point:  [0.01  0.01  0.035]
       Nodes:  [98 98 18]
       Spacing:  [0.00061856 0.00061856 0.00058824]


In [15]:
# 6. Calculate the CG fields
CG.fields_in_time()

 
-------------------- Calculating Coarse Grained Fields --------------------
 
---> Time step 0: time 0200 ................................................
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  15579
  Contact data loaded and mapped
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.001247377618865148 and search sampling factor 1000
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
  velocity done
  velocity gradient done
  kinetic tensor done
  contact tensor done
  particle density done
  total stress done
  d43 done
  d32 done
  Z done
  pressure done
  granular temp done
  shear rate done
  inertial number done
  mu done


/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/pysammos/coarse_graining.py:894: RuntimeWarning: divide by zero encountered in divide
  FrictionalCoefficient_Dxy_Py = TotalStressDeviator_xy_mag / Pressure_y
/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/pysammos/coarse_graining.py:896: RuntimeWarning: divide by zero encountered in divide
  FrictionalCoefficient_Dxyz_Pxy = TotalStressDeviator_xyz_mag / Pressure_xy
/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/pysammos/coarse_graining.py:897: RuntimeWarning: divide by zero encountered in divide
  FrictionalCoefficient_Dxyz_Py = TotalStressDeviator_xyz_mag / Pressure_y


  frabric tensor done
... fields computed
Writing results for timestep 200...
  File successfully updated to ./PysammosCG/CG_Lucy_Monodisperse.h5
  File successfully written to ./PysammosCG/CG_Lucy_Monodisperse_0200.vtkhdf
... results written
>> time step 0 took 19.774614095687866 to run.
  
